# SRQ generalization M7 - task-wise error trajectory

This train-only diagnostic follows the formal M6 width-20k retention failure. It measures factor, randomized system-action, weight, logit, prediction, and margin trajectories at locked widths 10k and 20k. It neither relaxes M6 nor selects a new method or width.

In [ ]:
# Edit repository/path values only. Scientific settings and gates are source-locked.
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH='experiment/soho-selfcontained'
WORK_DIR='/content/SOHO-CL'
FEATURE_CACHE_DIR='/content/srq_m7_cifar_features'
OUTPUT_DIR='/content/srq_m7_error_trajectory_output'
M6_ARTIFACT='/content/srq_generalization_m6_width_sweep_train_only.zip'
BATCH_SIZE=128
NUM_WORKERS=2

In [ ]:
# Fresh clone, dependencies, GPU check, and canonical-LF source verification.
import hashlib,json,os,shutil,subprocess,sys
from pathlib import Path
os.chdir('/content')
repo=Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR],check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub'],check=True)
import torch
assert torch.cuda.is_available(),'Select Runtime -> Change runtime type -> T4 GPU.'
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
EXPECTED={
 'configs/srq_generalization_m7_error_trajectory_train_only.json':'e8507b40e30af34677b36b34a8b9da18f2e49402f46e4604844813cd26ed38e4',
 'tools/srq_generalization_m7.py':'90d10de0a1b4b44d08b110c35a228271c943f8b355825e191f5b64c153919b9d',
 'tools/experiment_runner.py':'b2c953eea312a98ce4146757fb46a7dd0e4ebe20aa139da59464921b11f8310c',
 'methods/analytic_ridge/backends.py':'40114dcc05a9d991682f840efc48e7c74a3e12dde99be049a30ad5f825650dee',
 'methods/analytic_ridge/accounting.py':'a8aca83a0f2f9b2170e54b7739982fa150287939b2b8a7fe06db972196face05',
 'methods/analytic_ridge/compressed_upper.py':'dc3ff6ca1c62255628e40d8a9c29610c5ee138eadf5c352aa3b926c2cacd8f62',
 'methods/analytic_ridge/qr.py':'19d24d887e60ee7fc7adb1a2265253ea261c90636aef241241e5c6faad3a507b'}
for path,expected in EXPECTED.items(): assert sha(path)==expected,(path,sha(path),expected)
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip(),'Repository must start clean.'
CONFIG='configs/srq_generalization_m7_error_trajectory_train_only.json'
RUNNER='tools/srq_generalization_m7.py'
print('GPU:',torch.cuda.get_device_name(0))
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
print('SOURCE LOCK: PASS')

In [ ]:
# Fast local gates before downloading data.
command=[sys.executable,'-m','pytest','-q','tests/test_srq_generalization_m7.py','tests/test_srq_generalization_m6.py','tests/test_analytic_ridge_backend.py','tests/test_analytic_ridge_equivalence.py']
completed=subprocess.run(command)
assert completed.returncode==0,'M7 synthetic correctness gate failed; return the complete traceback.'
print('M7 SYNTHETIC GATE: PASS')

In [ ]:
# Upload and verify the immutable M6 input artifact.
from google.colab import files
m6_path=Path(M6_ARTIFACT)
if not m6_path.is_file():
    print('Upload srq_generalization_m6_width_sweep_train_only.zip')
    uploaded=files.upload()
    assert m6_path.name in uploaded,'Upload the exact M6 ZIP without renaming it.'
    m6_path.write_bytes(uploaded[m6_path.name])
assert sha(m6_path)=='b2739b9da023ebd2eedb6fdfe01c394e94f252773e847533b35350021c3d239e',(sha(m6_path),'wrong M6 artifact')
print('M6 ARTIFACT LOCK: PASS')

In [ ]:
# Download the locked backbone checkpoint and processed CIFAR-100 source.
import kagglehub
from huggingface_hub import hf_hub_download
CHECKPOINT_PATH=hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k',filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size==346284714
assert sha(CHECKPOINT_PATH)=='32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
CIFAR_ROOT=kagglehub.dataset_download('zaphat206/cifar-100')
print('CHECKPOINT:',CHECKPOINT_PATH)
print('CIFAR ROOT:',CIFAR_ROOT)

In [ ]:
# Materialize TRAIN features only; held-out test features remain absent.
cache=Path(FEATURE_CACHE_DIR)
if not (cache/'train.pt').is_file():
    command=[sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',CIFAR_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size','346284714','--backbone-checkpoint-sha256','32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b','--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir','/content/unused_m7','--dataset','CIFAR-100','--model-name','vit_base_patch16_224','--data-augmentation','vit','--seed','2025','--num-classes','100','--num-tasks','10','--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    print('TRAIN FEATURE EXTRACTION START',flush=True)
    subprocess.run(command,check=True)
assert (cache/'train.pt').is_file() and not (cache/'test.pt').exists()
metadata=json.loads((cache/'metadata.json').read_text())
assert metadata['feature_dim']==768 and metadata['finite'] is True
print('TRAIN CACHE READY:',metadata.get('train_shape'),'| test.pt absent')

In [ ]:
# Run Exact, FP16, and P2B sequentially at the two locked diagnostic widths.
command=[sys.executable,'-u',RUNNER,'run','--config',CONFIG,'--m6-artifact',M6_ARTIFACT,'--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir',OUTPUT_DIR,'--device','cuda','--require-clean-git']
print('M7 START: widths 10000/20000; task-wise factor/system/weight/logit/margin diagnosis.',flush=True)
completed=subprocess.run(command)
result_path=Path(OUTPUT_DIR)/'m7_results.json'
assert result_path.is_file(),'M7 failed before writing diagnostics; return the complete runner output.'
result=json.loads(result_path.read_text())
print('STATUS:',result['status'])
print('SUMMARY:',json.dumps(result['summary'],indent=2))
print('GATES:',json.dumps(result['gates'],indent=2))
assert completed.returncode==0 and result['status']=='PASS_M7_ERROR_TRAJECTORY_TRAIN_ONLY','M7 integrity/numerical gate failed; preserve output and do not tune or inspect test data.'

In [ ]:
# Human-readable trajectory table.
import pandas as pd
trajectory=pd.read_csv(Path(OUTPUT_DIR)/'error_trajectory.csv')
display(trajectory)

In [ ]:
# Create a source-derived vector diagnostic figure.
import matplotlib.pyplot as plt
fig,axes=plt.subplots(2,2,figsize=(10,7),sharex=True)
colors={10000:'#1f77b4',20000:'#d62728'}
for width in (10000,20000):
    rows=trajectory[(trajectory.width==width)&(trajectory.method=='p2b_int8')]
    label=f'{width//1000}k'
    axes[0,0].plot(rows.task,rows.relative_system_action_error,marker='o',label=label,color=colors[width])
    axes[0,1].plot(rows.task,rows.relative_weight_error,marker='o',label=label,color=colors[width])
    axes[1,0].plot(rows.task,rows.relative_logit_error,marker='o',label=label,color=colors[width])
    axes[1,1].plot(rows.task,1-rows.prediction_agreement,marker='o',label=label,color=colors[width])
axes[0,0].set_ylabel('Relative system-action error'); axes[0,1].set_ylabel('Relative weight error')
axes[1,0].set_ylabel('Relative logit error'); axes[1,1].set_ylabel('Prediction-change fraction')
for ax in axes.flat:
    ax.set_xlabel('Task'); ax.grid(True,alpha=.25); ax.legend()
for ax in (axes[0,0],axes[0,1],axes[1,0]): ax.set_yscale('log')
fig.suptitle('P2B task-wise approximation trajectory'); fig.tight_layout()
plot_path=Path(OUTPUT_DIR)/'m7_error_trajectory.svg'; fig.savefig(plot_path,format='svg'); plt.show(); plt.close(fig)
assert plot_path.is_file(); print('VECTOR FIGURE READY')

In [ ]:
# Export evidence only; feature cache, checkpoint, and M6 input are excluded.
bundle=Path('/content/srq_generalization_m7_error_trajectory_train_only')
if bundle.exists(): shutil.rmtree(bundle)
bundle.mkdir()
for source,name in [(Path(OUTPUT_DIR)/'m7_results.json','m7_results.json'),(Path(OUTPUT_DIR)/'error_trajectory.csv','error_trajectory.csv'),(Path(OUTPUT_DIR)/'m7_error_trajectory.svg','m7_error_trajectory.svg'),(Path(CONFIG),'config.json')]: shutil.copy2(source,bundle/name)
archive=shutil.make_archive(str(bundle),'zip',root_dir=bundle)
print('ARTIFACT:',archive,'sha256=',sha(archive))
files.download(archive)